# **🏠 House Prices: From Linear Models to Blending & Stacking**


> **This notebook documents my journey in the House Prices - Advanced Regression Techniques competition.The goal is to predict housing prices using tabular data with 80+ features.I started with linear models (Ridge, Lasso, ElasticNet), then tried tree-based models (XGBoost), and finally built ensemble methods (Blending & Stacking).The best submission achieved a Leaderboard RMSE of 0.127, placing in the top ~20%.**


In [ ]:
##Step 1 Import & Setup

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import KFold, cross_val_score
from sklearn.metrics import mean_squared_error
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import RidgeCV, LassoCV, ElasticNetCV

import xgboost as xgb

# RMSE scorer
def rmse(y_true, y_pred):
    return np.sqrt(mean_squared_error(y_true, y_pred))

In [ ]:
##Step 2 - Load Data

train = pd.read_csv("/kaggle/input/house-prices-advanced-regression-techniques/train.csv")
test = pd.read_csv("/kaggle/input/house-prices-advanced-regression-techniques/test.csv")

print("Train shape:", train.shape)
print("Test shape:", test.shape)

In [ ]:
##Step 3 - Target Variable

y = np.log1p(train["SalePrice"])
train_id = train["Id"]
test_id = test["Id"]

In [ ]:
# Step 4 - Combine train & test for consistent preprocessing
all_data = pd.concat([train.drop(["SalePrice", "Id"], axis=1),
                      test.drop("Id", axis=1)], axis=0)

# Fill missing values

# Numeric: fill with median
num_cols = all_data.select_dtypes(include=[np.number]).columns
all_data[num_cols] = all_data[num_cols].fillna(all_data[num_cols].median())

# Categorical: fill with mode (most frequent)
cat_cols = all_data.select_dtypes(exclude=[np.number]).columns
all_data[cat_cols] = all_data[cat_cols].fillna(all_data[cat_cols].mode().iloc[0])


# One-hot encode categoricals
all_data = pd.get_dummies(all_data)

# Separate back
ntrain = train.shape[0]
train_processed = all_data.iloc[:ntrain, :]
test_processed = all_data.iloc[ntrain:, :]

# Scale features for linear models
scaler = StandardScaler()
train_scaled = scaler.fit_transform(train_processed)
test_scaled = scaler.transform(test_processed)

**Linear Models**

In [ ]:
##Ridge Regression
alphas = [0.1, 1, 10, 50, 100, 200, 500]
ridge = RidgeCV(alphas=alphas, scoring="neg_mean_squared_error", cv=5)
ridge.fit(train_scaled, y)

print("Best alpha (Ridge):", ridge.alpha_)
ridge_rmse = np.sqrt(-ridge.best_score_)
print("CV RMSE (Ridge):", ridge_rmse)

In [ ]:
##Lasso Regression
from sklearn.linear_model import LassoCV, ElasticNetCV
from sklearn.model_selection import cross_val_score
import numpy as np

# ----- Lasso -----
lasso = LassoCV(
    alphas=np.logspace(-4, -0.5, 50),
    max_iter=50000,
    cv=5,
    random_state=42
)
lasso.fit(train_scaled, y)

print("Best alpha (Lasso):", lasso.alpha_)

# Use cross_val_score for reliable RMSE
lasso_rmse = np.sqrt(-cross_val_score(lasso, train_scaled, y, 
                                      cv=5, scoring="neg_mean_squared_error").mean())
print("CV RMSE (Lasso):", lasso_rmse)

In [ ]:
# ----- ElasticNet -----
elasticnet = ElasticNetCV(
    l1_ratio=[.1, .5, .7, .9, .95, .99, 1],  # test mix of L1/L2 ratios
    alphas=np.logspace(-4, -0.5, 50),
    max_iter=50000,
    cv=5,
    random_state=42
)
elasticnet.fit(train_scaled, y)

print("Best alpha (ElasticNet):", elasticnet.alpha_)
print("Best l1_ratio (ElasticNet):", elasticnet.l1_ratio_)

elasticnet_rmse = np.sqrt(-cross_val_score(elasticnet, train_scaled, y,
                                           cv=5, scoring="neg_mean_squared_error").mean())
print("CV RMSE (ElasticNet):", elasticnet_rmse)

**Tree Based Model**

In [ ]:
##XGBoost
xgb_model = xgb.XGBRegressor(
    n_estimators=2000,
    learning_rate=0.01,
    max_depth=4,
    subsample=0.7,
    colsample_bytree=0.8,
    reg_lambda=1,
    random_state=42,
    tree_method="hist"
)

kf = KFold(n_splits=5, shuffle=True, random_state=42)
cv_rmse = []
for tr_idx, val_idx in kf.split(train_processed):
    Xtr, Xval = train_processed.iloc[tr_idx], train_processed.iloc[val_idx]
    ytr, yval = y.iloc[tr_idx], y.iloc[val_idx]
    
    xgb_model.fit(Xtr, ytr, verbose=False)
    pred_val = xgb_model.predict(Xval)
    cv_rmse.append(rmse(yval, pred_val))

print("XGB 5-fold CV RMSE:", np.mean(cv_rmse), "+/-", np.std(cv_rmse))


In [ ]:
##Blending XGBoost and Lasso
lasso_preds = np.expm1(lasso.predict(test_scaled))
xgb_model.fit(train_processed, y)
xgb_preds = np.expm1(xgb_model.predict(test_processed))

# Blend with weight 0.35 Lasso + 0.65 XGB
blend_preds = 0.35 * lasso_preds + 0.65 * xgb_preds

pd.DataFrame({"Id": test_id, "SalePrice": blend_preds}).to_csv("submission_blend.csv", index=False)


In [ ]:
##Stacking with Meta-Model (Ridge)
from sklearn.linear_model import Ridge

# OOF predictions (simplified for notebook)
lasso_oof = lasso.predict(train_scaled)
xgb_oof = xgb_model.predict(train_processed)

stack_train = np.vstack([lasso_oof, xgb_oof]).T
stack_test = np.vstack([lasso_preds, xgb_preds]).T

meta = Ridge(alpha=1.0)
meta.fit(stack_train, y)
stack_preds = np.expm1(meta.predict(stack_test))

pd.DataFrame({"Id": test_id, "SalePrice": stack_preds}).to_csv("submission_stack.csv", index=False)